In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 시퀀스 길이와 히든 차원을 현실적으로 조정
batch_size, seq_len, input_size, hidden_size = 64, 5, 1, 4
torch.manual_seed(42)

# 과거 정보(T=0)와 현재 정보(T=-1)를 동시에 참조해야만 맞출 수 있는 XOR 형태의 데이터셋 구축
X_data = torch.randn(batch_size, seq_len, input_size)
# T=0이 양수고 T=-1이 양수면 1, 아니면 0 (두 시점을 다 봐야 함)
y_data = ((X_data[:, 0, 0] > 0) ^ (X_data[:, -1, 0] > 0)).long()

class GateTrackerLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        # 가중치 행렬 가로채기
        w_ih = self.lstm.weight_ih_l0
        return self.fc(out[:, -1, :]), w_ih

model = GateTrackerLSTM(input_size, hidden_size)
criterion = nn.CrossEntropyLoss()
# 아담 옵티마이저와 충분한 에폭으로 게이트를 강하게 자극
optimizer = optim.Adam(model.parameters(), lr=0.05)

print("=== [실험 1 v2] 게이트 가중치 자율 분화 추적 ===")

for epoch in range(201):
    optimizer.zero_grad()
    outputs, w_ih = model(X_data)
    loss = criterion(outputs, y_data)
    loss.backward()

    if epoch in [0, 100, 200]:
        # PyTorch LSTM W_ih 구조: [W_i, W_f, W_g, W_o] 순서로 Chunk
        w_i, w_f, _, _ = w_ih.grad.chunk(4, dim=0)

        print(f"\n[Epoch {epoch:3d}] Loss: {loss.item():.4f}")
        # 각 게이트 가중치 행렬이 학습되면서 그라디언트가 각자 다른 '스케일'과 '부호'로 움직임을 증명
        print(f" ├─ Input Gate  Grad (첫 2개 노드): {[f'{x:.6f}' for x in w_i[:2, 0].tolist()]}")
        print(f" └─ Forget Gate Grad (첫 2개 노드): {[f'{x:.6f}' for x in w_f[:2, 0].tolist()]}")

    optimizer.step()

=== [실험 1 v2] 게이트 가중치 자율 분화 추적 ===

[Epoch   0] Loss: 0.7245
 ├─ Input Gate  Grad (첫 2개 노드): ['-0.000123', '0.000730']
 └─ Forget Gate Grad (첫 2개 노드): ['-0.000017', '0.000307']

[Epoch 100] Loss: 0.0059
 ├─ Input Gate  Grad (첫 2개 노드): ['0.000171', '0.000010']
 └─ Forget Gate Grad (첫 2개 노드): ['-0.000106', '-0.000041']

[Epoch 200] Loss: 0.0015
 ├─ Input Gate  Grad (첫 2개 노드): ['0.000065', '0.000018']
 └─ Forget Gate Grad (첫 2개 노드): ['-0.000019', '0.000003']
